# Output of the symbolic tendencies: Land-Atmosphere model example 

In this notebook, we show how to create the symbolic tendencies of the model. Symbolic tendencies here means that it is possible to make any parameter of the model appears in the tendencies equations.

This can be done in several languages (Python, Julia, Fortran), but here, we are going to use Python.

More details about the model used in this notebook can be found in the articles:
* Hamilton, O., Demaeyer, J., Vannitsem, S., & Crucifix, M. (2025). *Using Unstable Periodic Orbits to Understand Blocking Behaviour in a Low Order Land-Atmosphere Model*. Submitted to Chaos. [preprint](https://doi.org/10.48550/arXiv.2503.02808)
* Xavier, A. K., Demaeyer, J., & Vannitsem, S. (2024). *Variability and predictability of a reduced-order land–atmosphere coupled model.* Earth System Dynamics, **15**(4), 893-912. [doi:10.5194/esd-15-893-2024](https://doi.org/10.5194/esd-15-893-2024)

or in the documentation.

## Modules import

First, setting the path and loading of some modules

In [1]:
import sys, os
sys.path.extend([os.path.abspath('../../')])

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from numba import njit

In [4]:
from scipy.integrate import solve_ivp

Importing the model's modules

In [5]:
from dapper.mods.Qgs.qgs.params.params import QgParams

In [6]:
from dapper.mods.Qgs.qgs.functions.symbolic_tendencies import create_symbolic_tendencies
from dapper.mods.Qgs.qgs.functions.tendencies import create_tendencies

## Systems definition

General parameters

In [7]:
# Time parameters
dt = 0.1

Setting some model parameters

In [8]:
# Model parameters instantiation with some non-default specs
model_parameters = QgParams()

# Mode truncation at the wavenumber 2 in both x and y spatial
# coordinates for the atmosphere
model_parameters.set_atmospheric_channel_fourier_modes(2, 2)
# Mode truncation at the wavenumber 2 in the x and at the 
# wavenumber 4 in the y spatial coordinates for the ocean
model_parameters.set_oceanic_basin_fourier_modes(2, 4)

In [9]:
# Setting MAOOAM parameters according to the publication linked above
model_parameters.set_params({'kd': 0.0290, 'kdp': 0.0290, 'n': 1.5, 'r': 1.e-7,
                             'h': 136.5, 'd': 1.1e-7})
model_parameters.atemperature_params.set_params({'eps': 0.7, 'T0': 289.3,
                                                 'hlambda': 15.06})
model_parameters.gotemperature_params.set_params({'gamma': 5.6e8, 'T0': 301.46})

Setting the short-wave radiation component as in the publication above: $C_{\text{a},1}$ and $C_{\text{o},1}$ 


In [10]:
model_parameters.atemperature_params.set_insolation(103.3333, 0)
model_parameters.gotemperature_params.set_insolation(310, 0)

Printing the model's parameters

In [11]:
# model_parameters.print_params()

Testing code for error insertion to tendency function in tendencies.py

In [12]:
from dapper.mods.Qgs.qgs.inner_products.analytic import AtmosphericAnalyticInnerProducts, OceanicAnalyticInnerProducts

aip = AtmosphericAnalyticInnerProducts(model_parameters)
oip = OceanicAnalyticInnerProducts(model_parameters)
aip.connect_to_ocean(oip)

In [13]:
# Printing the model's parameters
model_parameters.print_params()

Qgs v1.0.0 parameters summary

General Parameters:
'dynamic_T': False,
'T4': False,
'time_unit': days,
'rr': 287.058  [J][kg^-1][K^-1]  (gas constant of dry air),
'sb': 5.67e-08  [J][m^-2][s^-1][K^-4]  (Stefan-Boltzmann constant),

Scale Parameters:
'scale': 5000000.0  [m]  (characteristic space scale (L*pi)),
'f0': 0.0001032  [s^-1]  (Coriolis parameter at the middle of the domain),
'n': 1.5  [nondim]  (aspect ratio (n = 2 L_y / L_x)),
'rra': 6370000.0  [m]  (earth radius),
'phi0_npi': 0.25  [nondim]  (latitude expressed in fraction of pi),
'deltap': 50000.0  [Pa]  (pressure difference between the two atmospheric layers),
'Ha': 8500.0  [m]  (Average height of the 500 hPa pressure level at midlatitude),

Atmospheric Parameters:
'kd': 0.029  [nondim]  (atmosphere bottom friction coefficient),
'kdp': 0.029  [nondim]  (atmosphere internal friction coefficient),
'sigma': 0.2  [nondim]  (static stability of the atmosphere),

Atmospheric Temperature Parameters:
'gamma': 10000000.0  [J][m^-2]

## Outputting the model equations

Calculating the tendencies in Python as a function of the parameters $C_{{\rm g},0}$, $C_{{\rm a},0}$, $k_d$ and $k'_d$:

In [14]:
model_parameters.atmospheric_params

Atmospheric Parameters:
'kd': 0.029  [nondim]  (atmosphere bottom friction coefficient),
'kdp': 0.029  [nondim]  (atmosphere internal friction coefficient),
'sigma': 0.2  [nondim]  (static stability of the atmosphere),

In [15]:
model_parameters.atemperature_params

Atmospheric Temperature Parameters:
'gamma': 10000000.0  [J][m^-2][K^-1]  (specific heat capacity of the atmosphere),
'C[1]': 103.3333  [W][m^-2]  (spectral component 1 of the short-wave radiation of the atmosphere),
'C[2]': 0.0  [W][m^-2]  (spectral component 2 of the short-wave radiation of the atmosphere),
'C[3]': 0.0  [W][m^-2]  (spectral component 3 of the short-wave radiation of the atmosphere),
'C[4]': 0.0  [W][m^-2]  (spectral component 4 of the short-wave radiation of the atmosphere),
'C[5]': 0.0  [W][m^-2]  (spectral component 5 of the short-wave radiation of the atmosphere),
'C[6]': 0.0  [W][m^-2]  (spectral component 6 of the short-wave radiation of the atmosphere),
'C[7]': 0.0  [W][m^-2]  (spectral component 7 of the short-wave radiation of the atmosphere),
'C[8]': 0.0  [W][m^-2]  (spectral component 8 of the short-wave radiation of the atmosphere),
'C[9]': 0.0  [W][m^-2]  (spectral component 9 of the short-wave radiation of the atmosphere),
'C[10]': 0.0  [W][m^-2]  (spect

In [16]:
model_parameters.oceanic_params

Oceanic Parameters:
'gp': 0.031  [m][s^-2]  (reduced gravity),
'r': 1e-07  [s^-1]  (frictional coefficient at the bottom of the ocean),
'h': 136.5  [m]  (depth of the water layer of the ocean),
'd': 1.1e-07  [s^-1]  (strength of the ocean-atmosphere mechanical coupling),

In [17]:
model_parameters.gotemperature_params

Oceanic Temperature Parameters:
'gamma': 560000000.0  [J][m^-2][K^-1]  (specific heat capacity of the ocean),
'C[1]': 310.0  [W][m^-2]  (spectral component 1 of the short-wave radiation of the ocean),
'C[2]': 0.0  [W][m^-2]  (spectral component 2 of the short-wave radiation of the ocean),
'C[3]': 0.0  [W][m^-2]  (spectral component 3 of the short-wave radiation of the ocean),
'C[4]': 0.0  [W][m^-2]  (spectral component 4 of the short-wave radiation of the ocean),
'C[5]': 0.0  [W][m^-2]  (spectral component 5 of the short-wave radiation of the ocean),
'C[6]': 0.0  [W][m^-2]  (spectral component 6 of the short-wave radiation of the ocean),
'C[7]': 0.0  [W][m^-2]  (spectral component 7 of the short-wave radiation of the ocean),
'C[8]': 0.0  [W][m^-2]  (spectral component 8 of the short-wave radiation of the ocean),
'C[9]': 0.0  [W][m^-2]  (spectral component 9 of the short-wave radiation of the ocean),
'C[10]': 0.0  [W][m^-2]  (spectral component 10 of the short-wave radiation of the ocea

In [18]:
nonfixed_params = [
        model_parameters.atmospheric_params.kd,
        model_parameters.atmospheric_params.kdp,
        model_parameters.atmospheric_params.sigma,
        model_parameters.oceanic_params.gp,
        model_parameters.oceanic_params.r,
        model_parameters.oceanic_params.h,
        model_parameters.oceanic_params.d,
        model_parameters.atemperature_params.gamma,
        model_parameters.atemperature_params.C[0],
        model_parameters.atemperature_params.eps,
        model_parameters.atemperature_params.T0,
        model_parameters.atemperature_params.sc,
        model_parameters.atemperature_params.hlambda,
        model_parameters.gotemperature_params.gamma,
        model_parameters.gotemperature_params.C[0],
        model_parameters.gotemperature_params.T0
    ]

In [19]:
%%time
funcs, = create_symbolic_tendencies(
    model_parameters, 
    continuation_variables=nonfixed_params, 
    language='python'
    )

CPU times: user 3.58 s, sys: 270 ms, total: 3.85 s
Wall time: 57.5 s


Let's inspect the output:

In [21]:
print(funcs)

@njit
def f(t, U, k_d, k_p, sigma, g_p, r, h, d, gamma_a, C_a1, epsilon, T_a0, sc, lmda, gamma_o, C_go1, T_go0):
	# Tendency function of the qgs model
	# k_d:	atmosphere bottom friction coefficient
	# k_p:	atmosphere internal friction coefficient
	# sigma:	static stability of the atmosphere
	# g_p:	reduced gravity
	# r:	frictional coefficient at the bottom of the ocean
	# h:	depth of the water layer of the ocean
	# d:	strength of the ocean-atmosphere mechanical coupling
	# gamma_a:	specific heat capacity of the atmosphere
	# C_a1:	spectral component 1 of the short-wave radiation of the atmosphere
	# epsilon:	emissivity coefficient for the grey-body atmosphere
	# T_a0:	stationary solution for the 0-th order atmospheric temperature
	# sc:	ratio of surface to atmosphere temperature
	# lmda:	sensible+turbulent heat exchange between ocean/ground and atmosphere
	# gamma_o:	specific heat capacity of the ocean
	# C_go1:	spectral component 1 of the short-wave radiation of the ocean
	# T_go0:	st

Note that the tendencies have been already formatted as a [Numba](https://numba.pydata.org/) function, but it is easy to extract the equations for any other kind of accelerator or simply to produce pure Python code.

It is now easy to get the function into operation:

In [24]:
exec(funcs)

Save tendency function to new script

In [25]:
with open("tendencies_nonfixed_params.py", "w") as f_io:
    f_io.write(funcs)

and

In [26]:
# Parameters as defined above (in order to pass to tendency function)
default_params = (0.029, 0.029, 0.2, 0.031, 1e-7, 136.5, 1.1e-7, 1e7, 103.3333, 0.7, 
                  289.3, 1.0, 15.06, 5.6e8, 310, 301.46)

In [27]:
f(0.,np.zeros(model_parameters.ndim), *default_params)

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 4.84292694e-04, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 4.36192252e-05, 0.00000000e+00, 1.74476901e-05,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00])

## Comparing with numerical results

We can check that the symbolic (parameters dependent) equations are the same as the `qgs` numerical ones (with the same values of the parameters):

In [28]:
f_num, Df = create_tendencies(model_parameters)

In [29]:
f_num(0., np.zeros(model_parameters.ndim))

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 4.84292694e-04, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 4.36192252e-05, 0.00000000e+00, 1.74476901e-05,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00])

### See symbolic_output_land_atmosphere.ipynb for more (trajectory comparisons)